# Benchmark Tables
LaTeX tables for insertion and deletion experiments comparing DynSBOD with the
static algorithm.

In [1]:
from chart_utils import *
from collections import defaultdict

## Insertion Table

In [2]:
# Toggle to include second_half runs in the insertion table
INCLUDE_SECOND_HALF_IN_TABLE = True

static_datasets = (
    list((datasets_path / "complete").rglob("*.csv"))
    + list((datasets_path / "complete_long").rglob("*.csv"))
    + list((datasets_path / "wikipedia/dynfd").rglob("**/final_state.csv"))
)
static_datasets = [
    f for f in static_datasets
    if '_second_half' not in str(f) and '_first_half' not in str(f)
]


def parse_intermediate_results(identifier, include_second_half=False):
    """Parse DynSBOD intermediate results for a given identifier.

    Returns:
        tuple: (full_results, second_half_results) where each is a list of dicts
    """
    results_dir = dynsbod_root / "benchmark/dynsbod_results/intermediate"
    full_results = []
    second_half_results = []

    for file in results_dir.glob(f"*{identifier}*stdout.intermediate.csv"):
        file_str = str(file)
        is_second_half = '_second_half_' in file_str
        is_first_half = '_first_half_' in file_str

        if is_first_half:
            continue
        if is_second_half and not include_second_half:
            continue

        dataset_part = (
            file.name
            .replace('algo0_java0_', '')
            .replace('_stdout.intermediate.csv', '')
        )
        dataset_part = re.sub(r'_run\d+', '', dataset_part)
        dataset_part = dataset_part.replace('_first_half', '').replace('_second_half', '')

        if dataset_part != identifier:
            continue

        try:
            df = pd.read_csv(file)
            if len(df) > 0:
                last_row = df.iloc[-1]
                result = {
                    'constants': int(last_row['currentConstantOds']),
                    'compatibles': int(last_row['currentComptibleOds']),
                    'runtime': float(last_row['timeElapsedMs']) / 1000.0,
                    'memory': float(last_row['maxMemoryUsedBytes']) / (1024 * 1024),
                }
                if is_second_half:
                    second_half_results.append(result)
                else:
                    full_results.append(result)
        except Exception as e:
            print(f"Error parsing {file}: {e}")

    return full_results, second_half_results


# Build the table data
table_data = []

for dataset_path in static_datasets:
    dataset_name = get_dataset_name(dataset_path)

    delimiter = ';' if any(
        name in dataset_name.lower() for name in ['horse', 'adult', 'plista']
    ) else ','

    rows, cols = count_csv_dimensions(dataset_path, delimiter=delimiter)

    full_name = dataset_path.stem
    if full_name == 'final_state':
        identifier = f"{dataset_path.parent.name}_final_state"
    else:
        identifier = full_name

    static_results, static_error = parse_static_results(identifier)
    dynsbod_full_results, dynsbod_second_half_results = parse_intermediate_results(
        identifier, include_second_half=INCLUDE_SECOND_HALF_IN_TABLE
    )

    # Static averages
    if static_results:
        avg_static_time = (
            sum(r['time'] for r in static_results if r['time'] is not None)
            / len([r for r in static_results if r['time'] is not None])
            if any(r['time'] is not None for r in static_results)
            else None
        )
        avg_static_memory = (
            sum(r['memory'] for r in static_results if r['memory'] is not None)
            / len([r for r in static_results if r['memory'] is not None])
            if any(r['memory'] is not None for r in static_results)
            else None
        )
    else:
        avg_static_time = None
        avg_static_memory = None

    static_time_display = avg_static_time
    static_memory_display = avg_static_memory
    if avg_static_time is None and static_error:
        static_time_display = static_error.upper()
    if avg_static_memory is None and static_error:
        static_memory_display = static_error.upper()

    # DynSBOD full averages
    if dynsbod_full_results:
        avg_constants = sum(r['constants'] for r in dynsbod_full_results) / len(dynsbod_full_results)
        avg_compatibles = sum(r['compatibles'] for r in dynsbod_full_results) / len(dynsbod_full_results)
        avg_dynsbod_time = sum(r['runtime'] for r in dynsbod_full_results) / len(dynsbod_full_results)
        avg_dynsbod_memory = sum(r['memory'] for r in dynsbod_full_results) / len(dynsbod_full_results)
    else:
        avg_constants = avg_compatibles = avg_dynsbod_time = avg_dynsbod_memory = None

    # DynSBOD second-half averages
    if INCLUDE_SECOND_HALF_IN_TABLE and dynsbod_second_half_results:
        avg_dynsbod_second_half_time = (
            sum(r['runtime'] for r in dynsbod_second_half_results) / len(dynsbod_second_half_results)
        )
        avg_dynsbod_second_half_memory = (
            sum(r['memory'] for r in dynsbod_second_half_results) / len(dynsbod_second_half_results)
        )
    else:
        avg_dynsbod_second_half_time = None
        avg_dynsbod_second_half_memory = None

    row_data = {
        'Dataset': dataset_name,
        'Rows': rows,
        'Columns': cols,
        'Constants': avg_constants,
        'Compatibles': avg_compatibles,
        'Static Time (s)': static_time_display,
        'Static Memory (MB)': static_memory_display,
        'DynSBOD Time (s)': avg_dynsbod_time,
        'DynSBOD Memory (MB)': avg_dynsbod_memory,
    }

    if INCLUDE_SECOND_HALF_IN_TABLE:
        row_data['DynSBOD 2nd Half Time (s)'] = avg_dynsbod_second_half_time
        row_data['DynSBOD 2nd Half Memory (MB)'] = avg_dynsbod_second_half_memory

    table_data.append(row_data)

insertion_table = pd.DataFrame(table_data)

# ── Format for LaTeX ──
latex_table_df = insertion_table.copy()

for idx, row in latex_table_df.iterrows():
    static_time = row['Static Time (s)']
    static_mem = row['Static Memory (MB)']
    dynsbod_time = row['DynSBOD Time (s)']
    dynsbod_mem = row['DynSBOD Memory (MB)']

    latex_table_df.at[idx, 'Static Time (s)'] = format_numeric(static_time)
    latex_table_df.at[idx, 'Static Memory (MB)'] = format_numeric(static_mem)

    for col in ['Rows', 'Columns', 'Constants', 'Compatibles']:
        latex_table_df.at[idx, col] = format_numeric(row[col])

    latex_table_df.at[idx, 'DynSBOD Time (s)'] = format_with_bold(
        dynsbod_time, should_bold(dynsbod_time, static_time, lambda a, b: a < b)
    )
    latex_table_df.at[idx, 'DynSBOD Memory (MB)'] = format_with_bold(
        dynsbod_mem, should_bold(dynsbod_mem, static_mem, lambda a, b: a < b)
    )

    if INCLUDE_SECOND_HALF_IN_TABLE:
        dynsbod_2nd_time = row.get('DynSBOD 2nd Half Time (s)')
        dynsbod_2nd_mem = row.get('DynSBOD 2nd Half Memory (MB)')
        latex_table_df.at[idx, 'DynSBOD 2nd Half Time (s)'] = format_with_bold(
            dynsbod_2nd_time, should_bold(dynsbod_2nd_time, static_time, lambda a, b: a < b)
        )
        latex_table_df.at[idx, 'DynSBOD 2nd Half Memory (MB)'] = format_with_bold(
            dynsbod_2nd_mem, should_bold(dynsbod_2nd_mem, static_mem, lambda a, b: a < b)
        )

latex_table_df['Dataset'] = latex_table_df['Dataset'].str.replace('_', r'\_', regex=False)

latex_table = latex_table_df.to_latex(index=False, na_rep='-', escape=False)
print("\n\nLaTeX Table:")
print(latex_table)

with open(thesis_charts_path / 'insertion_table.tex', 'w') as f:
    f.write(latex_table)

insertion_table

static search for identifier='abalone' yielded results=[{'memory': 812.0, 'time': 1.918}, {'memory': 897.0, 'time': 1.832}, {'memory': 1036.0, 'time': 1.816}] error_type=None
static search for identifier='adult' yielded results=[{'memory': 3974.0, 'time': 138.814}, {'memory': 4287.0, 'time': 117.136}] error_type=None
static search for identifier='hepatitis' yielded results=[{'memory': 3868.0, 'time': 57.655}, {'memory': 2007.0, 'time': 58.841}, {'memory': 840.0, 'time': 57.57}] error_type=None
static search for identifier='ncvoter-reduced' yielded results=[{'memory': 12022.0, 'time': 379.376}] error_type=None
static search for identifier='letter' yielded results=[{'memory': 8657.0, 'time': 1871.599}, {'memory': 7961.0, 'time': 2049.158}, {'memory': 8018.0, 'time': 1907.772}] error_type=None
static search for identifier='iris' yielded results=[{'memory': 12.0, 'time': 0.034}, {'memory': 12.0, 'time': 0.033}, {'memory': 12.0, 'time': 0.038}] error_type=None
static search for identifier='

/var/folders/6w/1dx0z2b91j798m812cq_1d8m0000gn/T/ipykernel_93637/1286540712.py:168: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4177' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  latex_table_df.at[idx, col] = format_numeric(row[col])
/var/folders/6w/1dx0z2b91j798m812cq_1d8m0000gn/T/ipykernel_93637/1286540712.py:168: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '9' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  latex_table_df.at[idx, col] = format_numeric(row[col])
/var/folders/6w/1dx0z2b91j798m812cq_1d8m0000gn/T/ipykernel_93637/1286540712.py:168: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '137.00' has dtype incompatible with float64, please explicitly c

,Dataset,Rows,Columns,Constants,Compatibles,Static Time (s),Static Memory (MB),DynSBOD Time (s),DynSBOD Memory (MB),DynSBOD 2nd Half Time (s),DynSBOD 2nd Half Memory (MB)
0,abalone,4177,9,137.0,324.0,1.855333,915.0,2.135000,779.712504,1.088,363.975121
1,adult,32561,15,78.0,1419.0,127.975,4130.5,60.699500,13405.057800,31.011,10016.699615
2,hepatitis,156,20,8250.0,65751.0,58.022,2238.333333,10.738333,2083.196139,8.608,1805.370697
3,ncvoter-reduced,938086,15,58.0,353.0,379.376,12022.0,633.230000,21511.420990,205.224,21511.420311
4,letter,20000,17,61.0,2816.0,1942.843,8212.0,2157.419000,23420.502739,303.883,21513.057800
5,iris,150,5,4.0,10.0,0.035,12.0,0.137000,60.620588,0.085,58.686043
6,chess,28057,7,1.0,0.0,0.2965,54.0,0.699000,141.542760,0.417,101.327957
7,bridges,108,13,144.0,1213.0,0.662,314.333333,0.719000,96.596062,0.422,96.418251
8,flights-reduced,500001,12,39.0,33.0,None,None,533.823000,4360.686218,542.086,4123.458786
9,fd-reduced-30,250001,30,89571.0,759.0,OOM,OOM,9094.951000,25471.360756,4351.849,21558.024323


## Deletion Table

In [3]:
def parse_deletion_experiment(dataset_name, run_number, delete_dir):
    """Parse a single algo0 deletion experiment run."""
    try:
        stdout_file = delete_dir / f"algo0_java0_{dataset_name}_run{run_number}_stdout.txt"
        if not stdout_file.exists():
            return None

        with open(stdout_file, 'r') as f:
            content = f.read()

        stats = parse_deletion_stdout_stats(stdout_file)
        completed = stats is not None

        delete_csv = delete_dir / f"algo0_java0_{dataset_name}_run{run_number}_stdout.intermediate.csv_delete.csv"
        if delete_csv.exists():
            delete_df = pd.read_csv(delete_csv)
            if len(delete_df) > 0:
                delete_last = delete_df.iloc[-1]
                time_s = float(delete_last['timeElapsedMs']) / 1000.0
                memory_mb = float(delete_last['maxMemoryUsedBytes']) / (1024 * 1024)

                if completed and stats:
                    num_validations = stats['validationCount']
                    num_revalidations = stats['normalRevalidations'] + stats['directRevalidations']
                    pct_violations = 100 - (num_revalidations / num_validations * 100) if num_validations > 0 else 0
                    return {
                        'time': time_s,
                        'memory': memory_mb,
                        'num_validations': num_validations,
                        'pct_violations': pct_violations,
                        'ctx_steps_new': stats['contextIteratorStepsNew'],
                        'ctx_steps_without_merging': stats['contextIteratorStepsWithoutMerging'],
                        'completed': True,
                    }

        progress, oom = parse_progress_bar(content)
        if progress:
            return {
                'time': None, 'memory': None,
                'num_validations': None, 'pct_violations': None,
                'ctx_steps_new': None, 'ctx_steps_without_merging': None,
                'completed': False, 'progress_pct': progress['percent'], 'oom': oom,
            }

    except Exception as e:
        print(f"Error parsing algo0 {dataset_name} run{run_number}: {e}")

    return None


def parse_all_deletion_experiments():
    """Parse all deletion experiment data for algo0."""
    delete_dir = benchmark_data_path / "dynsbod_results/intermediate_delete"
    results = defaultdict(list)

    datasets = set()
    for file in delete_dir.glob("algo0_java0_*_run*_stdout.txt"):
        match = re.search(r'algo0_java0_(.+?)_run(\d+)_stdout\.txt', file.name)
        if match:
            datasets.add(match.group(1))

    for dataset_name in datasets:
        for run_num in [1, 2]:
            result = parse_deletion_experiment(dataset_name, run_num, delete_dir)
            if result:
                results[dataset_name].append(result)

    return results


def create_deletion_table(results, insertion_table):
    """Create deletion experiment summary table aligned with insertion table."""
    table_data = []
    insertion_datasets = insertion_table['Dataset'].tolist()

    name_mapping = {
        'ncvoter': 'ncvoter_22_1m_utf8',
        'flights': 'flights_20_500k',
        'plista': 'plista_1k',
    }

    insertion_time_lookup = {}
    for _, irow in insertion_table.iterrows():
        ds = irow['Dataset']
        t = irow.get('DynSBOD Time (s)')
        if isinstance(t, (int, float)) and pd.notna(t):
            insertion_time_lookup[ds] = t

    for dataset_name in insertion_datasets:
        deletion_dataset = name_mapping.get(dataset_name, dataset_name)
        if deletion_dataset not in results:
            continue

        algo_runs = results[deletion_dataset]
        if not algo_runs:
            continue

        row_data = {'Dataset': dataset_name}
        row_data['Insert Time (s)'] = insertion_time_lookup.get(dataset_name, None)

        completed_runs = [r for r in algo_runs if r['completed']]
        incomplete_runs = [r for r in algo_runs if not r['completed']]

        if completed_runs:
            avg_time = sum(r['time'] for r in completed_runs) / len(completed_runs)
            avg_memory = sum(r['memory'] for r in completed_runs) / len(completed_runs)
            avg_validations = int(sum(r['num_validations'] for r in completed_runs) / len(completed_runs))
            avg_pct_violations = sum(r['pct_violations'] for r in completed_runs) / len(completed_runs)
            avg_ctx_steps_new = int(sum(r['ctx_steps_new'] for r in completed_runs) / len(completed_runs))
            avg_ctx_steps_without_merging = int(sum(r['ctx_steps_without_merging'] for r in completed_runs) / len(completed_runs))

            row_data['Delete Time (s)'] = avg_time
            row_data['Delete Mem (MB)'] = avg_memory
            row_data['Validations'] = avg_validations
            row_data['%Violations'] = avg_pct_violations
            row_data['CtxStepsNew'] = avg_ctx_steps_new
            row_data['CtxStepsWithoutMerging'] = avg_ctx_steps_without_merging
        elif incomplete_runs:
            avg_progress = sum(r['progress_pct'] for r in incomplete_runs) / len(incomplete_runs)
            any_oom = any(r.get('oom', False) for r in incomplete_runs)
            status = f"OOM@{avg_progress:.0f}\\%" if any_oom else f"TO@{avg_progress:.0f}\\%"
            row_data['Delete Time (s)'] = status
            row_data['Delete Mem (MB)'] = 'OOM' if any_oom else 'TO'
            row_data['Validations'] = None
            row_data['%Violations'] = None
            row_data['CtxStepsNew'] = None
            row_data['CtxStepsWithoutMerging'] = None

        table_data.append(row_data)

    return pd.DataFrame(table_data)


# ── Build deletion table ──
deletion_results = parse_all_deletion_experiments()
deletion_table = create_deletion_table(deletion_results, insertion_table)

latex_deletion_table = deletion_table.copy()
# Convert numeric columns to object so we can replace with formatted strings
for _c in ['Insert Time (s)', 'Delete Time (s)', 'Delete Mem (MB)', 'Validations', '%Violations', 'CtxStepsNew', 'CtxStepsWithoutMerging']:
    if _c in latex_deletion_table.columns:
        latex_deletion_table[_c] = latex_deletion_table[_c].astype(object)

for idx, row in latex_deletion_table.iterrows():
    for col in ['Insert Time (s)', 'Delete Time (s)', 'Delete Mem (MB)']:
        if col in row and isinstance(row[col], (int, float)) and pd.notna(row[col]):
            latex_deletion_table.at[idx, col] = f"{row[col]:.2f}"
    for col in ['Validations', 'CtxStepsNew', 'CtxStepsWithoutMerging']:
        if col in row and isinstance(row[col], (int, float)) and pd.notna(row[col]):
            latex_deletion_table.at[idx, col] = f"{int(row[col])}"
    if '%Violations' in row and isinstance(row['%Violations'], (int, float)) and pd.notna(row['%Violations']):
        latex_deletion_table.at[idx, '%Violations'] = f"{row['%Violations']:.1f}\\%"

latex_deletion_table['Dataset'] = latex_deletion_table['Dataset'].str.replace('_', r'\_', regex=False)

latex_deletion_output = latex_deletion_table.to_latex(index=False, na_rep='-', escape=False)

print("\n\nDeletion Experiment Table:")
print(latex_deletion_output)

with open(thesis_charts_path / 'deletion_table.tex', 'w') as f:
    f.write(latex_deletion_output)

deletion_table



Deletion Experiment Table:
\begin{tabular}{llllllll}
\toprule
Dataset & Insert Time (s) & Delete Time (s) & Delete Mem (MB) & Validations & %Violations & CtxStepsNew & CtxStepsWithoutMerging \\
\midrule
abalone & 2.13 & 1.33 & 665.81 & 13174 & 58.5\% & 5073 & 6316 \\
adult & 60.70 & 131.21 & 3542.47 & 1309037 & 1.7\% & 1101585 & 3448569 \\
hepatitis & 10.74 & 1085.11 & 2397.13 & 19401603 & 0.2\% & 6092266 & 26544939 \\
ncvoter-reduced & 633.23 & 779.61 & 16107.10 & 564322 & 2.8\% & 331836 & 953156 \\
letter & 2157.42 & 958.53 & 6121.96 & 8879118 & 4.5\% & 6415766 & 23751918 \\
iris & 0.14 & 0.19 & 68.90 & 343 & 73.8\% & 354 & 458 \\
chess & 0.70 & 0.88 & 188.97 & 3119 & 65.0\% & 3867 & 4887 \\
bridges & 0.72 & 1.67 & 711.12 & 83500 & 6.7\% & 36090 & 83492 \\
fd-reduced-30 & 9094.95 & 1899.87 & 10159.48 & 1791346 & 49.3\% & 54747 & 83963 \\
ncvoter & 22286.18 & OOM@4\% & OOM & - & - & - & - \\
flights & 3004.76 & OOM@4\% & OOM & - & - & - & - \\
plista & 7003.10 & TO@0\% & TO & - & - 

,Dataset,Insert Time (s),Delete Time (s),Delete Mem (MB),Validations,%Violations,CtxStepsNew,CtxStepsWithoutMerging
0,abalone,2.135000,1.3305,665.805504,13174.0,58.486413,5073.0,6316.0
1,adult,60.699500,131.208,3542.466511,1309037.0,1.703237,1101585.0,3448569.0
2,hepatitis,10.738333,1085.1145,2397.131371,19401603.0,0.240887,6092266.0,26544939.0
3,ncvoter-reduced,633.230000,779.6085,16107.100388,564322.0,2.752329,331836.0,953156.0
4,letter,2157.419000,958.528,6121.955914,8879118.0,4.527781,6415766.0,23751918.0
5,iris,0.137000,0.1915,68.900948,343.0,73.760933,354.0,458.0
6,chess,0.699000,0.882,188.966553,3119.0,64.956717,3867.0,4887.0
7,bridges,0.719000,1.667,711.115929,83500.0,6.714970,36090.0,83492.0
8,fd-reduced-30,9094.951000,1899.868,10159.477562,1791346.0,49.319283,54747.0,83963.0
9,ncvoter,22286.180333,OOM@4\%,OOM,NaN,NaN,NaN,NaN


In [4]:

# Ratio: Validations during deletion vs. ODs holding after full insertion
# ODs holding after insertion = Constants + Compatibles (from insertion_table)

ratio_df = deletion_table[['Dataset', 'Validations']].copy()

# Bring in Constants, Compatibles, and Rows from insertion_table
ins_ods = insertion_table[['Dataset', 'Constants', 'Compatibles', 'Rows']].copy()
ratio_df = ratio_df.merge(ins_ods, on='Dataset', how='left')

ratio_df['Total ODs'] = ratio_df['Constants'] + ratio_df['Compatibles']
ratio_df['Validations / Total ODs'] = ratio_df.apply(
    lambda r: (r['Validations'] / r['Total ODs'])
    if pd.notna(r['Validations']) and pd.notna(r['Total ODs']) and r['Total ODs'] > 0
    else None,
    axis=1,
)
ratio_df['Validations / (Total ODs * Rows)'] = ratio_df.apply(
    lambda r: (r['Validations'] / (r['Total ODs'] * r['Rows']))
    if pd.notna(r['Validations']) and pd.notna(r['Total ODs']) and pd.notna(r['Rows'])
       and r['Total ODs'] > 0 and r['Rows'] > 0
    else None,
    axis=1,
)

display_df = ratio_df[['Dataset', 'Validations', 'Total ODs', 'Rows',
                        'Validations / Total ODs', 'Validations / (Total ODs * Rows)']].copy()
display_df['Validations'] = display_df['Validations'].apply(
    lambda v: f"{int(v):,}" if pd.notna(v) else '-'
)
display_df['Total ODs'] = display_df['Total ODs'].apply(
    lambda v: f"{int(v):,}" if pd.notna(v) else '-'
)
display_df['Rows'] = display_df['Rows'].apply(
    lambda v: f"{int(v):,}" if pd.notna(v) else '-'
)
display_df['Validations / Total ODs'] = display_df['Validations / Total ODs'].apply(
    lambda v: f"{v:.4f}" if pd.notna(v) else '-'
)
display_df['Validations / (Total ODs * Rows)'] = display_df['Validations / (Total ODs * Rows)'].apply(
    lambda v: f"{v:.6f}" if pd.notna(v) else '-'
)

print("Validations (deletion) vs. ODs holding after full insertion:\n")
print(display_df.to_string(index=False))
display_df


Validations (deletion) vs. ODs holding after full insertion:

            Dataset Validations Total ODs    Rows Validations / Total ODs Validations / (Total ODs * Rows)
            abalone      13,174       461   4,177                 28.5770                         0.006842
              adult   1,309,037     1,497  32,561                874.4402                         0.026855
          hepatitis  19,401,603    74,001     156                262.1803                         1.680643
    ncvoter-reduced     564,322       411 938,086               1373.0462                         0.001464
             letter   8,879,118     2,877  20,000               3086.2419                         0.154312
               iris         343        14     150                 24.5000                         0.163333
              chess       3,119         1  28,057               3119.0000                         0.111167
            bridges      83,500     1,357     108                 61.5328         

,Dataset,Validations,Total ODs,Rows,Validations / Total ODs,Validations / (Total ODs * Rows)
0,abalone,"13,174",461,"4,177",28.5770,0.006842
1,adult,"1,309,037","1,497","32,561",874.4402,0.026855
2,hepatitis,"19,401,603","74,001",156,262.1803,1.680643
3,ncvoter-reduced,"564,322",411,"938,086",1373.0462,0.001464
4,letter,"8,879,118","2,877","20,000",3086.2419,0.154312
5,iris,343,14,150,24.5000,0.163333
6,chess,"3,119",1,"28,057",3119.0000,0.111167
7,bridges,"83,500","1,357",108,61.5328,0.569748
8,fd-reduced-30,"1,791,346","90,330","250,001",19.8311,0.000079
9,ncvoter,-,"17,401","938,085",-,-
